In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("disease_prediction.csv")
df.head()

,patient_id,age,gender,glucose_mg_dl,cholesterol_mg_dl,systolic_bp,diastolic_bp,bmi,heart_rate,smoking,alcohol_consumption,physical_activity,family_history,disease
0,1,32,Male,101,235,152,79,28.5,73,No,Yes,Low,Yes,Yes
1,2,31,Male,124,191,134,77,33.9,71,No,Yes,Low,Yes,Yes
2,3,45,Male,57,141,114,71,27.2,79,Yes,Yes,Low,No,No
3,4,75,Female,69,268,120,82,21.5,61,Yes,Yes,Medium,No,Yes
4,5,53,Male,107,163,131,75,23.3,73,Yes,No,Low,Yes,Yes


In [49]:
df['physical_activity'].value_counts()

physical_activity
Medium    344
High      344
Low       312
Name: count, dtype: int64

In [4]:
df.isnull().sum()

patient_id             0
age                    0
gender                 0
glucose_mg_dl          0
cholesterol_mg_dl      0
systolic_bp            0
diastolic_bp           0
bmi                    0
heart_rate             0
smoking                0
alcohol_consumption    0
physical_activity      0
family_history         0
disease                0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   patient_id           1000 non-null   int64  
 1   age                  1000 non-null   int64  
 2   gender               1000 non-null   object 
 3   glucose_mg_dl        1000 non-null   int64  
 4   cholesterol_mg_dl    1000 non-null   int64  
 5   systolic_bp          1000 non-null   int64  
 6   diastolic_bp         1000 non-null   int64  
 7   bmi                  1000 non-null   float64
 8   heart_rate           1000 non-null   int64  
 9   smoking              1000 non-null   object 
 10  alcohol_consumption  1000 non-null   object 
 11  physical_activity    1000 non-null   object 
 12  family_history       1000 non-null   object 
 13  disease              1000 non-null   object 
dtypes: float64(1), int64(7), object(6)
memory usage: 109.5+ KB


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder
from sklearn.linear_model import LogisticRegression  

In [52]:
onehot=['gender','smoking','alcohol_consumption','family_history']
tr1=ColumnTransformer(transformers=[
    ('tr1',OneHotEncoder(drop='first'),onehot),
    ('tr2',OrdinalEncoder(categories=[['Low','Medium','High']]),['physical_activity'])
],remainder='passthrough')

In [91]:
x=df.iloc[:,1:13]
y=df.iloc[:,-1]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [92]:
leb=LabelEncoder()
leb.fit(y_train)
y_train_encode=leb.transform(y_train)
y_test_encode=leb.transform(y_test)

In [93]:
x_train_encode=tr1.fit_transform(x_train)
x_test_encode=tr1.transform(x_test)

In [94]:
from sklearn.metrics import accuracy_score,precision_score,f1_score

In [108]:
lr=LogisticRegression(max_iter=10000)
lr.fit(x_train_encode,y_train_encode)
lr_pred=lr.predict(x_test_encode)
print(accuracy_score(y_test_encode,lr_pred))
print(precision_score(y_test_encode,lr_pred))
print(f1_score(y_test_encode,lr_pred))

0.865
0.8695652173913043
0.8556149732620321


In [109]:
# So we will try to elemenate the columns which are less importance
lr.coef_

array([[ 0.02123705,  2.20679499,  0.97028586,  2.35478115, -1.76585732,
         0.05400029,  0.03710588,  0.01532615,  0.04211956, -0.00491794,
         0.16864705, -0.02470567]])

In [110]:
feature_names = tr1.get_feature_names_out()

for feature, coef in zip(feature_names, lr.coef_[0]):
    print(feature, coef)

tr1__gender_Male 0.021237051023476332
tr1__smoking_Yes 2.2067949869523136
tr1__alcohol_consumption_Yes 0.9702858570452753
tr1__family_history_Yes 2.354781148494843
tr2__physical_activity -1.76585732224755
remainder__age 0.05400029017727529
remainder__glucose_mg_dl 0.037105882086925154
remainder__cholesterol_mg_dl 0.015326146289846042
remainder__systolic_bp 0.042119558928201344
remainder__diastolic_bp -0.004917936529748106
remainder__bmi 0.16864704640299233
remainder__heart_rate -0.02470567016541315


In [111]:
# so we already removed the pasenger id because it has not contributed to our prediction
# we can also see thst smpking and faly history playes an imortant role in predicting the result
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [132]:
rf=RandomForestClassifier()
rf.fit(x_train_encode,y_train_encode)
rf_pred=rf.predict(x_test_encode)
print("accuracy_score: ",accuracy_score(y_test_encode,rf_pred))
print("precision_score: ",precision_score(y_test_encode,rf_pred))
print("f1_score: ",f1_score(y_test_encode,rf_pred))

accuracy_score:  0.94
precision_score:  0.9191919191919192
f1_score:  0.9381443298969072


In [131]:
import xgboost as xgb
xg = xgb.XGBClassifier(
     n_estimators=1000,
    learning_rate=0.09,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8
)
xg.fit(x_train_encode,y_train_encode)
xg_pred=xg.predict(x_test_encode)
print("accuracy_score: ",accuracy_score(y_test_encode,xg_pred))
print("precision_score: ",precision_score(y_test_encode,xg_pred))
print("f1_score: ",f1_score(y_test_encode,xg_pred))

accuracy_score:  0.99
precision_score:  0.9894736842105263
f1_score:  0.9894736842105263


In [135]:
model = Pipeline([
    ('preprocessing', tr1),
    ('xgboost', xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.09,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])

model.fit(x_train,y_train_encode)

y_pred = model.predict(x_test)

print("accuracy_score:", accuracy_score(y_test_encode, y_pred))
print("precision_score:", precision_score(y_test_encode, y_pred))
print("f1_score:", f1_score(y_test_encode, y_pred))

accuracy_score: 0.99
precision_score: 0.9894736842105263
f1_score: 0.9894736842105263
